# 05 - MusicBrainz Artist Enrichment

## Goal

Enrich Ticketmaster artists with MusicBrainz metadata to support artist matching and concert recommendation.

## Tasks

- Extract unique artists from PostgreSQL
- Query MusicBrainz for artist matches
- Store canonical artist identifiers
- Review ambiguous or missing matches
- Prepare enriched artist data for the ranking engine

In [ ]:
import os
from pathlib import Path
import pandas as pd
import requests

from dotenv import load_dotenv
from sqlalchemy import URL, text, create_engine

In [ ]:
project_path = Path("..")

load_dotenv(project_path / ".env", override= True)
db_user = os.getenv("POSTGRES_USER")
db_password = os.getenv("POSTGRES_PASSWORD")
db_name = os.getenv("POSTGRES_DB")
db_port = os.getenv("POSTGRES_PORT")

assert db_user is not None
assert db_password is not None
assert db_name is not None
assert db_port is not None

In [ ]:
database_url = URL.create(
    drivername= "postgresql+psycopg2",
    username= db_user,
    password= db_password,
    host= "localhost",
    port= int(db_port),
    database= db_name
)

engine= create_engine(database_url)

In [ ]:
artists_df = pd.read_sql(
    """
    SELECT
        DISTINCT artist_name
    FROM events
    WHERE artist_name IS NOT NULL
    ORDER BY artist_name
    """,
    engine
)

artists_df.shape

In [ ]:
test_artist = (artists_df).iloc[0]["artist_name"]

params = {
    "query": f'artist:"{test_artist}"',
    "fmt": "json",
    "limit": 5
}
response = requests.get(
    musicbrainz_url,
    params=params,
    headers=headers,
    timeout=30
)

response.status_code

In [ ]:
artist_data = response.json()
artist_data.keys()

artist_data.get('artists', [])[:2]

In [ ]:
import time
import json

## MusicBrainz Configuration

MusicBrainz requires applications to identify themselves with a meaningful User-Agent and limits clients to one request per second.

A local cache is used to avoid repeating API requests when the notebook is rerun.

In [ ]:
musicbrainz_url = "https://musicbrainz.org/ws/2/artist/"

headers = {
    "User-Agent": "GigRouteEurope/1.0"
}

In [ ]:
cache_path = (
    project_path
    / "data"
    / "raw"
    / "musicbrainz"
    / "artist_search_cache.json"
)

cache_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

In [ ]:
if cache_path.exists():
    with open(cache_path, "r", encoding="utf-8") as file:
        artist_cache = json.load(file)
else:
    artist_cache = {}

print("Cached artists:", len(artist_cache))

In [ ]:
def search_musicbrainz_artist(
    artist_name,
    session,
    limit=5,
    max_retries=3
):
    params = {
        "query": f'artist:"{artist_name}"',
        "fmt": "json",
        "limit": limit
    }

    for attempt in range(1, max_retries + 1):
        try:
            response = session.get(
                musicbrainz_url,
                params=params,
                headers=headers,
                timeout=45
            )

            if response.status_code == 200:
                data = response.json()

                return {
                    "status_code": 200,
                    "artists": data.get("artists", [])
                }

            if response.status_code in [429, 502, 503, 504]:
                wait_seconds = attempt * 5

                print(
                    f"Temporary API error {response.status_code} "
                    f"for {artist_name}. "
                    f"Retrying in {wait_seconds}s..."
                )

                time.sleep(wait_seconds)
                continue

            return {
                "status_code": response.status_code,
                "artists": []
            }

        except requests.exceptions.RequestException:
            wait_seconds = attempt * 5

            print(
                f"Request failed for {artist_name}. "
                f"Attempt {attempt}/{max_retries}. "
                f"Retrying in {wait_seconds}s..."
            )

            time.sleep(wait_seconds)

    return {
        "status_code": None,
        "artists": []
    }

In [ ]:
artist_names = artists_df["artist_name"].dropna().unique()

print("Unique artists to process:", len(artist_names))

In [54]:
session = requests.Session()

for index, artist_name in enumerate(artist_names, start=1):
    cached_result = artist_cache.get(artist_name)

    if (
        cached_result is not None
        and cached_result.get("status_code") == 200
    ):
        print(
            f"{index}/{len(artist_names)} "
            f"| Cached | {artist_name}"
        )
        continue

    result = search_musicbrainz_artist(
        artist_name,
        session
    )

    artist_cache[artist_name] = result

    with open(
        cache_path,
        "w",
        encoding="utf-8"
    ) as file:
        json.dump(
            artist_cache,
            file,
            ensure_ascii=False,
            indent=2
        )

    print(
        f"{index}/{len(artist_names)} "
        f"| Status {result['status_code']} "
        f"| {artist_name} "
        f"| Candidates: {len(result['artists'])}"
    )

    time.sleep(1.1)

1/472 | Status 200 | 54 Ultra | Candidates: 1
2/472 | Cached | 6LACK
3/472 | Cached | 8lanco
4/472 | Status 200 | 90s Super Show | Candidates: 0
5/472 | Cached | A$AP Rocky
6/472 | Cached | Abbamania
7/472 | Status 200 | aespa | Candidates: 1
8/472 | Cached | Against Evil
9/472 | Cached | Aleks Syntek
10/472 | Cached | Alex Spencer
11/472 | Status 200 | Allie Sherlock | Candidates: 1
12/472 | Cached | Altered Rebirth
13/472 | Cached | Amble
14/472 | Status 200 | Amelie Lens | Candidates: 1
15/472 | Status 200 | Amorphis | Candidates: 2
16/472 | Cached | Anastacia
17/472 | Status 200 | And One | Candidates: 5


KeyboardInterrupt: 

In [21]:
print("Ticketmaster artists:", len(artist_names))
print("Cached MusicBrainz searches:", len(artist_cache))

Ticketmaster artists: 472
Cached MusicBrainz searches: 472


## Build Candidate Dataset

Cached MusicBrainz responses are flattened into a tabular candidate dataset for entity matching.

In [60]:
candidate_rows = []

for artist_name, result in artist_cache.items():

    candidates = result.get("artists", [])

    for rank, candidate in enumerate(candidates, start=1):

        candidate_rows.append({
            "ticketmaster_artist_name": artist_name,
            "candidate_rank": rank,
            "mbid": candidate.get("id"),
            "musicbrainz_name": candidate.get("name"),
            "sort_name": candidate.get("sort-name"),
            "artist_type": candidate.get("type"),
            "country": candidate.get("country"),
            "disambiguation": candidate.get("disambiguation"),
            "score": candidate.get("score")
        })

In [61]:
candidates_df = pd.DataFrame(candidate_rows)
candidates_df.head()

candidates_df.shape
candidates_df.columns

candidates_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 833 entries, 0 to 832
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   ticketmaster_artist_name  833 non-null    str  
 1   candidate_rank            833 non-null    int64
 2   mbid                      833 non-null    str  
 3   musicbrainz_name          833 non-null    str  
 4   sort_name                 833 non-null    str  
 5   artist_type               747 non-null    str  
 6   country                   501 non-null    str  
 7   disambiguation            444 non-null    str  
 8   score                     833 non-null    int64
dtypes: int64(2), str(7)
memory usage: 58.7 KB


In [46]:
candidates_df["score"] = pd.to_numeric(
    candidates_df["score"],
    errors="coerce"
)

candidates_df.dtypes

ticketmaster_artist_name      str
candidate_rank              int64
mbid                          str
musicbrainz_name              str
sort_name                     str
artist_type                   str
country                       str
disambiguation                str
score                       int64
dtype: object

In [47]:
artists_with_candidates = (
    candidates_df["ticketmaster_artist_name"]
    .nunique()
)

print("Ticketmaster artists:", len(artist_names))
print("Artists with candidates:", artists_with_candidates)

Ticketmaster artists: 472
Artists with candidates: 349


In [48]:
matched_names = set(
    candidates_df["ticketmaster_artist_name"]
)

artists_without_candidates = [
    artist_name
    for artist_name in artist_names
    if artist_name not in matched_names
]

print(
    "Artists without candidates:",
    len(artists_without_candidates)
)

Artists without candidates: 123


In [49]:
artists_without_candidates[:20]

['54 Ultra',
 '90s Super Show',
 'aespa',
 'Allie Sherlock',
 'Amelie Lens',
 'Amorphis',
 'And One',
 'Andreas Gabalier',
 'Apache 207',
 'Ash',
 'Austra',
 'Baby Keem',
 'Bagjan Oktyabr',
 'Bailey Zimmerman',
 'BAP – Die Zielgerade',
 'Bayou',
 'Beat It! - Das Musical über den King of Pop',
 'Beatrice Egli',
 'BENNETT',
 'Ben Wood Inferno']

In [50]:
failed_searches = {
    artist_name: result
    for artist_name, result in artist_cache.items()
    if result.get("status_code") != 200
}

print("Failed searches:", len(failed_searches))

Failed searches: 61


In [52]:
from collections import Counter

In [53]:
failed_statuses = Counter(
    result.get("status_code")
    for result in failed_searches.values()
)

failed_statuses

Counter({503: 61})

## Retry Failed Searches

Previously failed MusicBrainz requests are retried separately so that temporary network or API errors are not treated as missing artist matches.

In [55]:
failed_artist_names = list(failed_searches.keys())

print("Artists to retry:", len(failed_artist_names))

Artists to retry: 61


In [56]:
for index, artist_name in enumerate(
    failed_artist_names,
    start=1
):
    result = search_musicbrainz_artist(
        artist_name,
        session
    )

    artist_cache[artist_name] = result

    with open(
        cache_path,
        "w",
        encoding="utf-8"
    ) as file:
        json.dump(
            artist_cache,
            file,
            ensure_ascii=False,
            indent=2
        )

    print(
        f"{index}/{len(failed_artist_names)} "
        f"| Status {result['status_code']} "
        f"| {artist_name} "
        f"| Candidates: {len(result['artists'])}"
    )

    time.sleep(1.1)

1/61 | Status 200 | 54 Ultra | Candidates: 1
2/61 | Status 200 | 90s Super Show | Candidates: 0
3/61 | Status 200 | aespa | Candidates: 1
4/61 | Status 200 | Allie Sherlock | Candidates: 1
5/61 | Status 200 | Amelie Lens | Candidates: 1
6/61 | Status 200 | Amorphis | Candidates: 2
7/61 | Status 200 | And One | Candidates: 5
8/61 | Status 200 | Andreas Gabalier | Candidates: 1
9/61 | Status 200 | Apache 207 | Candidates: 1
Request failed for Ash. Attempt 1/3. Retrying in 5s...
10/61 | Status 200 | Ash | Candidates: 5
11/61 | Status 200 | Austra | Candidates: 5
12/61 | Status 200 | Baby Keem | Candidates: 1
Request failed for Bagjan Oktyabr. Attempt 1/3. Retrying in 5s...
13/61 | Status 200 | Bagjan Oktyabr | Candidates: 0
Request failed for Bailey Zimmerman. Attempt 1/3. Retrying in 5s...
14/61 | Status 200 | Bailey Zimmerman | Candidates: 1
15/61 | Status 200 | Bayou | Candidates: 5
16/61 | Status 200 | Beatrice Egli | Candidates: 1
17/61 | Status 200 | BENNETT | Candidates: 5
18/61 | 

In [ ]:
failed_searches = {
    artist_name: result
    for artist_name, result in artist_cache.items()
    if result.get("status_code") != 200
}

print("Failed searches after retry:", len(failed_searches))

Failed searches after retry: 0


In [ ]:
candidates_df.shape

(833, 9)

In [63]:
matched_names = set(
    candidates_df["ticketmaster_artist_name"]
)

artists_without_candidates = [
    artist_name
    for artist_name in artist_names
    if artist_name not in matched_names
]

print(
    "Artists without candidates:",
    len(artists_without_candidates)
)

artists_without_candidates[:20]

Artists without candidates: 72


['90s Super Show',
 'Bagjan Oktyabr',
 'BAP – Die Zielgerade',
 'Beat It! - Das Musical über den King of Pop',
 'Ben Wood Inferno',
 'Berlin, Du coole Sau',
 'Bloc Party & Interpol',
 'Bühne Frei mit Mariska Nijhof & KMT',
 'CA7RIEL & Paco Amoroso',
 'CLB100',
 'Curtis on Tour',
 'DarioM & Friends',
 'Das MOVEMENT ART – International Dance Festival',
 'Delil & AVIE',
 'Die besten Comedians Deutschlands',
 'Die Liga Der Gewoehnlichen Gentlemen',
 'Die Musik von Ennio Morricone - Lords of the Sound',
 'Drei Haselnüsse für Aschenbrödel - Das Musical',
 'DWEDA Records',
 'Eifeler Blechfestival']

In [68]:
top_candidates = (
    candidates_df.sort_values(["ticketmaster_artist_name", "score"],
                              ascending=[True, False])
                  .drop_duplicates(subset="ticketmaster_artist_name", keep= "first")
                  .reset_index(drop= True)
)
top_candidates.head()

,ticketmaster_artist_name,candidate_rank,mbid,musicbrainz_name,sort_name,artist_type,country,disambiguation,score
0,54 Ultra,1,6f4926ae-865e-483b-86f9-9ae3b9584507,54 Ultra,54 Ultra,Person,NaN,NaN,100
1,6LACK,1,07832b42-8826-4ab1-acd3-c49a2f595ffe,6LACK,6LACK,Person,US,NaN,100
2,8lanco,1,641c819b-d57c-43c1-8bff-9e66f5fd2feb,8lanco,8lanco,Person,NO,NaN,100
3,A$AP Rocky,1,25b7b584-d952-4662-a8b9-dd8cdfbfeb64,A$AP Rocky,ASAP Rocky,Person,US,US rapper,100
4,Abbamania,1,bd2219b2-a506-4726-87e0-6406323b0d47,ABBAmania Canada,ABBAmania Canada,Group,CA,"Canadian ABBA tribute, not to be confused w/UK...",100


In [69]:
top_candidates["score"].describe()

count    400.0
mean     100.0
std        0.0
min      100.0
25%      100.0
50%      100.0
75%      100.0
max      100.0
Name: score, dtype: float64

### Top Candidate Scores

All top-ranked MusicBrainz candidates received a score of 100.

Because the top score does not distinguish match confidence in this dataset, candidate ambiguity is evaluated by comparing the first and second-ranked candidate scores.

## Candidate Ambiguity

The first and second-ranked MusicBrainz candidates are compared to identify artist searches where multiple candidates have similarly strong scores.

In [75]:
score_comparison = (
    candidates_df
    .pivot_table(
        index="ticketmaster_artist_name",
        columns="candidate_rank",
        values="score",
        aggfunc="first"
    )
)

score_comparison.head()

candidate_rank,1,2,3,4,5
ticketmaster_artist_name,,,,,
54 Ultra,100.0,NaN,NaN,NaN,NaN
6LACK,100.0,NaN,NaN,NaN,NaN
8lanco,100.0,NaN,NaN,NaN,NaN
A$AP Rocky,100.0,NaN,NaN,NaN,NaN
Abbamania,100.0,NaN,NaN,NaN,NaN


In [76]:
candidate_counts = (
    candidates_df
    .groupby("ticketmaster_artist_name")
    .size()
    .value_counts()
    .sort_index()
)

candidate_counts

1    251
2     37
3     19
4     14
5     79
Name: count, dtype: int64

In [77]:
multiple_candidate_scores = score_comparison[
    score_comparison[2].notna()
].copy()

In [78]:
multiple_candidate_scores["score_gap"] = (
    multiple_candidate_scores[1]
    - multiple_candidate_scores[2]
)

In [79]:
multiple_candidate_scores["score_gap"].describe()

count    149.000000
mean      14.194631
std       10.468536
min        0.000000
25%        5.000000
50%       13.000000
75%       22.000000
max       49.000000
Name: score_gap, dtype: float64

In [80]:
multiple_candidate_scores.sort_values("score_gap").head(20)

candidate_rank,1,2,3,4,5,score_gap
ticketmaster_artist_name,,,,,,
Ben Ellis,100.0,100.0,100.0,100.0,NaN,0.0
Cannelle,100.0,99.0,98.0,97.0,90.0,1.0
Cave,100.0,99.0,94.0,86.0,86.0,1.0
Eloise,100.0,99.0,97.0,95.0,95.0,1.0
KELS,100.0,99.0,98.0,98.0,89.0,1.0
TJARK,100.0,99.0,98.0,89.0,89.0,1.0
Lori,100.0,99.0,99.0,98.0,96.0,1.0
Kilimanjaro,100.0,99.0,99.0,97.0,90.0,1.0
Razors,100.0,99.0,98.0,97.0,94.0,1.0


### Ambiguity Threshold

Among artists with multiple MusicBrainz candidates, the 25th percentile of the score gap between the first and second candidates is 5.

A score gap of 5 or less is therefore used as an exploratory threshold to flag potentially ambiguous matches for further entity-resolution review.

In [82]:
ambiguous_artists = (
    multiple_candidate_scores[
        multiple_candidate_scores["score_gap"] <= 5
    ]
    .sort_values("score_gap")
)

print("Ambiguous artists:", len(ambiguous_artists))

ambiguous_artists.head(20)

Ambiguous artists: 38


candidate_rank,1,2,3,4,5,score_gap
ticketmaster_artist_name,,,,,,
Ben Ellis,100.0,100.0,100.0,100.0,NaN,0.0
Cannelle,100.0,99.0,98.0,97.0,90.0,1.0
Cave,100.0,99.0,94.0,86.0,86.0,1.0
Eloise,100.0,99.0,97.0,95.0,95.0,1.0
Lori,100.0,99.0,99.0,98.0,96.0,1.0
Razors,100.0,99.0,98.0,97.0,94.0,1.0
KELS,100.0,99.0,98.0,98.0,89.0,1.0
Kilimanjaro,100.0,99.0,99.0,97.0,90.0,1.0
Wahnsinn!,100.0,99.0,96.0,94.0,94.0,1.0


In [83]:
ambiguous_names = ambiguous_artists.index

ambiguous_candidates = (
    candidates_df[
        candidates_df["ticketmaster_artist_name"].isin(ambiguous_names)
    ]
    .sort_values(
        ["ticketmaster_artist_name", "candidate_rank"]
    )
)

ambiguous_candidates[
    [
        "ticketmaster_artist_name",
        "candidate_rank",
        "musicbrainz_name",
        "artist_type",
        "country",
        "score",
        "disambiguation"
    ]
].head(50)

,ticketmaster_artist_name,candidate_rank,musicbrainz_name,artist_type,country,score,disambiguation
10,Alex Spencer,1,Alex Spencer,Person,NaN,100,Singer-songwriter from Manchester
11,Alex Spencer,2,Alex Spencer,Person,US,97,NaN
14,Amble,1,Amble,Group,IE,100,Trio of Irish contemporary folk musicians
15,Amble,2,Amble,NaN,NaN,95,electronic music
16,Amble,3,amble,Person,NaN,93,"Pianist, composer, improviser, Melbourne, Aust..."
17,Amble,4,Amble Skuse,Person,NaN,87,NaN
18,Amble,5,Carl August Amble Eriksen,Person,NaN,75,NaN
43,Ash,1,Ash,Group,GB,100,Northern Ireland alternative rock band
44,Ash,2,Wishbone Ash,Group,GB,97,NaN
45,Ash,3,Dragon Ash,Group,JP,82,NaN


In [84]:
processed_musicbrainz_path = (
    project_path
    / "data"
    / "processed"
    / "musicbrainz"
)

processed_musicbrainz_path.mkdir(
    parents=True,
    exist_ok=True
)

candidate_file = (
    processed_musicbrainz_path
    / "artist_candidates.csv"
)

candidates_df.to_csv(
    candidate_file,
    index=False
)

print("Saved:", candidate_file.exists())

Saved: True


In [85]:
ticketmaster_artist_count = len(artist_names)

artists_with_candidates = (
    candidates_df["ticketmaster_artist_name"]
    .nunique()
)

artists_without_candidates_count = (
    ticketmaster_artist_count
    - artists_with_candidates
)

single_candidate_artists = (
    candidate_counts.get(1, 0)
)

multiple_candidate_artists = (
    candidate_counts.drop(
        labels=1,
        errors="ignore"
    ).sum()
)

ambiguous_artist_count = len(ambiguous_artists)

total_candidate_rows = len(candidates_df)

In [86]:
print("Ticketmaster artists:", ticketmaster_artist_count)
print("Artists with candidates:", artists_with_candidates)
print(
    "Artists without candidates:",
    artists_without_candidates_count
)
print(
    "Single-candidate artists:",
    single_candidate_artists
)
print(
    "Multiple-candidate artists:",
    multiple_candidate_artists
)
print(
    "Ambiguous artists:",
    ambiguous_artist_count
)
print(
    "Total candidate rows:",
    total_candidate_rows
)

Ticketmaster artists: 472
Artists with candidates: 400
Artists without candidates: 72
Single-candidate artists: 251
Multiple-candidate artists: 149
Ambiguous artists: 38
Total candidate rows: 833
